# Lab 2: Local structure around a customer

The **account view** sees one customer's totals: money in, money out, number of payments. The **network view** also sees *who is on the other side*: how many distinct payers and payees, whether the same counterparties appear on both sides, whether money leaves as fast as it arrives.

Several patterns are invisible in the account view and obvious in the network view: **funnels**, **pass-through** accounts and **round trips**. We'll find them with simple 1-hop measures, then use timing and amounts to tell the real cases from legitimate look-alikes.

> **Shape finds candidates; context decides.** Everything here produces a list to review, not a verdict.

In [ ]:
# Setup: the last three months, hubs removed from the counterparties
import numpy as np, pandas as pd, networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import labkit as lk

nodes, edges = lk.load()
name, sector, region = (dict(zip(nodes.node_id, nodes[c])) for c in ("name", "sector", "region"))
HUBS = lk.find_hubs(edges)
WINDOW = ["2026-06", "2026-07", "2026-08"]
BEFORE = ["2025-12", "2026-01", "2026-02"]
win = edges[edges.month.isin(WINDOW) & ~edges.src.isin(HUBS) & ~edges.dst.isin(HUBS)]
pairs = win.groupby(["src", "dst"], as_index=False)[["amount", "n_txn"]].sum()
Gw = nx.DiGraph()
Gw.add_edges_from(zip(pairs.src, pairs.dst))
display(f"{len(HUBS)} hubs removed; window {WINDOW[0]} to {WINDOW[-1]}: {len(pairs):,} payer -> payee pairs")

## 1. Spot the pattern

Four business accounts. First the account view: does anything stand out?

In [ ]:
# The account view: totals only
ACCOUNTS = {"Account A": "B0403", "Account B": "B0858", "Account C": "B1982", "Account D": "B1300"}
rows = []
for label, a in ACCOUNTS.items():
    i, o = win[win.dst == a], win[win.src == a]
    rows.append({"account": label, "sector": sector[a], "received": lk.usd(i.amount.sum()), "payments in": int(i.n_txn.sum()),
                 "sent": lk.usd(o.amount.sum()), "payments out": int(o.n_txn.sum()),
                 "average payment": lk.usd((i.amount.sum() + o.amount.sum()) / max(1, i.n_txn.sum() + o.n_txn.sum()))})
pd.DataFrame(rows)

Now the network view of the same four accounts: payers on the left (blue lines = money in), payees on the right (orange = money out), counterparties on **both** sides at the bottom. Small grey dots are individuals; larger dots are businesses.

**Before you scroll on**, decide which account is a normal business, which is a funnel, which is a pass-through and which is round-tripping money.

In [ ]:
# The network view: one ego network per account
fig = make_subplots(rows=2, cols=2, subplot_titles=list(ACCOUNTS), horizontal_spacing=0.03, vertical_spacing=0.07)
for k, (label, a) in enumerate(ACCOUNTS.items()):
    r, c = k // 2 + 1, k % 2 + 1
    ego = Gw.subgraph({a} | set(Gw.predecessors(a)) | set(Gw.successors(a)))
    ego = nx.DiGraph([(u, v) for u, v in ego.edges() if a in (u, v)])
    pos = lk.ego_layout(ego, a)
    for (u, v) in ego.edges():
        both = ego.has_edge(v, u)
        off = 0.03 if both else 0.0
        clr = lk.OUT_COLOR if u == a else lk.IN_COLOR
        p, q = pos[u], pos[v]
        nrm = np.array([-(q - p)[1], (q - p)[0]]) / (np.linalg.norm(q - p) + 1e-9) * off * (1 if u == a else -1)
        fig.add_trace(go.Scatter(x=[p[0] + nrm[0], q[0] + nrm[0]], y=[p[1] + nrm[1], q[1] + nrm[1]], mode="lines",
                                 line=dict(width=1.4, color=clr), hoverinfo="skip", showlegend=False), r, c)
    others = [n for n in ego if n != a]
    fig.add_trace(go.Scatter(
        x=[pos[n][0] for n in others], y=[pos[n][1] for n in others], mode="markers", showlegend=False,
        marker=dict(size=[6 if n.startswith("P") else 12 for n in others],
                    color=[lk.OTHER if n.startswith("P") else lk.PALETTE[6] for n in others]),
        hovertext=[f"{n}: {name[n]}" for n in others], hoverinfo="text"), r, c)
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(size=22, color=lk.INK), showlegend=False,
                             hovertext=label, hoverinfo="text"), r, c)
fig.update_xaxes(visible=False, range=[-1.25, 1.25])
fig.update_yaxes(visible=False, range=[-1.15, 1.15])
lk.style(fig, "Same accounts, network view: blue = money in, orange = money out, bottom = both directions", height=900)

## 2. Measures you can compute from one hop

For each business, with **P** = distinct payers and **Q** = distinct payees over the window:

| measure | definition | what it points at |
|---|---|---|
| in-degree, out-degree | size of P, size of Q | breadth on each side |
| reciprocal counterparties r | size of P ∩ Q = in + out − distinct counterparties | partners on both sides (1-hop loops) |
| **overlap coefficient** | r / min(in, out) | 1 means almost every partner on the smaller side is also on the other side |
| fan-in ratio | (in + 1) / (out + 1) | many in, few out: a funnel shape |
| throughput | min(money in, money out) / max(money in, money out) | near 1: money leaves as it arrives |
| individual payer share | payers who are people / in-degree | businesses paid mainly by individuals |
| top payee share | largest payee / money out | concentration of outflow |
| loop symmetry | per reciprocal partner, min(out, back) / max(out, back) | near 1: what goes out comes back |

The overlap coefficient is exactly the "distinct counterparties close to max(in, out)" idea: if distinct ≈ max(in, out), then r ≈ min(in, out). Guard it with min(in, out) ≥ 3, or one payer who is also the one payee scores a perfect 1.

In [ ]:
# Local metrics for every business
out_ = pairs.groupby("src").agg(out_deg=("dst", "nunique"), out_amt=("amount", "sum"), top_payee=("amount", "max"))
in_ = pairs.groupby("dst").agg(in_deg=("src", "nunique"), in_amt=("amount", "sum"),
                               ind_payers=("src", lambda s: s.str.startswith("P").sum()))
back = pairs.merge(pairs, left_on=["src", "dst"], right_on=["dst", "src"], suffixes=("", "_back"))
back["symmetry"] = np.minimum(back.amount, back.amount_back) / np.maximum(back.amount, back.amount_back)
loops = back.groupby("src").agg(reciprocal=("dst", "nunique"), loop_symmetry=("symmetry", "median"))

L = in_.join(out_, how="outer").join(loops, how="left")
L = L[L.index.str.startswith("B")].fillna({"in_deg": 0, "out_deg": 0, "in_amt": 0, "out_amt": 0,
                                           "ind_payers": 0, "top_payee": 0, "reciprocal": 0})
small_side = L[["in_deg", "out_deg"]].min(1)
L["overlap"] = np.where(small_side > 0, L.reciprocal / small_side.replace(0, np.nan), np.nan)
L["fan_in_ratio"] = (L.in_deg + 1) / (L.out_deg + 1)
L["throughput"] = np.minimum(L.in_amt, L.out_amt) / np.maximum(L.in_amt, L.out_amt).replace(0, np.nan)
L["individual_payer_share"] = L.ind_payers / L.in_deg.replace(0, np.nan)
L["top_payee_share"] = L.top_payee / L.out_amt.replace(0, np.nan)
L["volume"] = L.in_amt + L.out_amt

before = edges[edges.month.isin(BEFORE) & ~edges.src.isin(HUBS)]
L["ind_payers_before"] = before[before.src.str.startswith("P")].groupby("dst").src.nunique().reindex(L.index).fillna(0)
L["in_deg_before"] = before.groupby("dst").src.nunique().reindex(L.index).fillna(0)

cols = ["in_deg", "out_deg", "reciprocal", "overlap", "fan_in_ratio", "throughput",
        "individual_payer_share", "top_payee_share", "loop_symmetry"]
spot = L.loc[list(ACCOUNTS.values()), cols].round(2)
spot.index = list(ACCOUNTS)
spot

**Reveal.** A is a pass-through (many in, many out, money leaves as fast as it comes), B is an ordinary business, C round-trips money with the same counterparties (overlap 1, symmetric loops), D is a funnel (dozens of individuals in, one payee out). The account view showed four ordinary totals.

## 3. The behavioural map

Every business on one chart. Corners are where the patterns live.

In [ ]:
# Behavioural map: fan-in ratio vs overlap coefficient, colour = throughput
M = L[(L.in_deg + L.out_deg) >= 3].copy()
rng = np.random.default_rng(1)
M["x"] = np.log2(M.fan_in_ratio)
M["y"] = M.overlap.fillna(0) + rng.normal(0, 0.012, len(M))
fig = go.Figure(go.Scatter(
    x=M.x, y=M.y, mode="markers",
    marker=dict(size=4 + 2.2 * np.log10(M.volume + 10), color=M.throughput.fillna(0), cmin=0, cmax=1,
                colorscale=[[0, "#cde2fb"], [0.5, "#5598e7"], [1, "#0d366b"]], showscale=True,
                colorbar=dict(title="throughput"), line=dict(width=0.5, color=lk.SURFACE)),
    hovertext=[f"{i}: {name[i]}<br>{sector[i]}<br>in {r.in_deg:.0f}, out {r.out_deg:.0f}, reciprocal {r.reciprocal:.0f}"
               f"<br>throughput {r.throughput:.2f}" for i, r in M.iterrows()], hoverinfo="text"))
fig.add_annotation(x=M.x.max() * 0.85, y=0.06, text="funnel corner: many in, few out", showarrow=False, font=dict(color=lk.MUTED))
fig.add_annotation(x=0, y=1.08, text="loop band: same partners on both sides", showarrow=False, font=dict(color=lk.MUTED))
fig.update_layout(xaxis_title="fan-in ratio, log2 (right = many payers, few payees)", yaxis_title="overlap coefficient")
lk.style(fig, "Every business, one hop: corners hold the unusual shapes", height=560)

In [ ]:
# Shape rules: each produces a candidate list
rules = {
    "funnel shape": (L.in_deg >= 20) & (L.out_deg <= 8) & (L.individual_payer_share >= 0.8) & (L.throughput >= 0.7),
    "pass-through shape": (L.in_deg >= 5) & (L.out_deg >= 5) & (L.throughput >= 0.9),
    "loop shape": (L.overlap >= 0.8) & (L[["in_deg", "out_deg"]].min(1) >= 3),
}
L["shape"] = ""
for k, m in rules.items():
    L.loc[m & (L["shape"] == ""), "shape"] = k
pd.DataFrame({"rule": list(rules), "businesses flagged": [int(m.sum()) for m in rules.values()]})

Too many pass-through candidates: plenty of ordinary businesses spend roughly what they earn. Shape alone is not enough. Two more pieces of evidence settle most cases:

* **Timing** (daily transactions for 2026-08): how much of the money that arrives leaves again within 5 days? Matched first in, first out.
* **History**: did the pattern exist six months ago, or did it start recently?
* For loops, **amounts and lags**: does what goes out come back at nearly the same amount, days later?

In [ ]:
# Timing evidence from daily transactions
tx = lk.load_transactions()

def fast_out_share(node, days=5):
    """Share of incoming dollars that leave again within `days` days (first in, first out)."""
    ins = tx[tx.dst == node].sort_values("date")[["date", "amount"]].to_numpy()
    outs = tx[tx.src == node].sort_values("date")[["date", "amount"]].to_numpy()
    if len(ins) == 0 or len(outs) == 0:
        return np.nan
    queue = [[d, a] for d, a in ins]
    total, matched, i = sum(a for _, a in ins), 0.0, 0
    for d_out, a_out in outs:
        while a_out > 1e-9 and i < len(queue) and queue[i][0] <= d_out:
            take = min(queue[i][1], a_out)
            if (d_out - queue[i][0]).days <= days:
                matched += take
            queue[i][1] -= take
            a_out -= take
            if queue[i][1] <= 1e-9:
                i += 1
    return matched / total

def return_lag_days(node):
    """Median days between paying a partner and that partner paying back (reciprocal partners only)."""
    lags = []
    for p in set(tx[tx.src == node].dst) & set(tx[tx.dst == node].src):
        outs = tx[(tx.src == node) & (tx.dst == p)].date
        backs = tx[(tx.src == p) & (tx.dst == node)].date
        for d in outs:
            later = backs[backs >= d]
            if len(later):
                lags.append((later.min() - d).days)
    return np.median(lags) if lags else np.nan

cand = L[L["shape"] != ""].copy()
cand["fast_out_5d"] = [fast_out_share(n) for n in cand.index]
cand["return_lag_days"] = [return_lag_days(n) if s == "loop shape" else np.nan for n, s in zip(cand.index, cand["shape"])]
cand["pattern_is_new"] = (cand.ind_payers_before < 0.5 * cand.ind_payers) | (cand.in_deg_before < 0.5 * cand.in_deg)
display(cand.groupby("shape")[["fast_out_5d", "loop_symmetry", "return_lag_days"]].describe().round(2).T)

In [ ]:
# Evidence rules on top of the shape
ev = cand.copy()
ev["evidence"] = np.select(
    [ev["shape"].eq("funnel shape") & (ev.fast_out_5d >= 0.6) & ev.pattern_is_new,
     ev["shape"].eq("pass-through shape") & (ev.fast_out_5d >= 0.8),
     ev["shape"].eq("loop shape") & (ev.loop_symmetry >= 0.85) & (ev.return_lag_days <= 7)],
    ["fast sweeps, new pattern", "money out within days", "symmetric loops, quick return"], default="")
ev["review"] = ev.evidence != ""
summary = ev.groupby("shape").agg(flagged_by_shape=("review", "size"), still_flagged_with_evidence=("review", "sum"))
display(summary, ev[ev.review][["shape", "evidence", "in_deg", "out_deg", "throughput", "fast_out_5d",
                                "loop_symmetry", "return_lag_days"]].round(2)
        .assign(name=lambda d: [name[i] for i in d.index]))

## 4. Check against the planted answers

The synthetic data comes with the truth. Legitimate look-alikes are planted too: **property managers** (dozens of tenants in, owners and contractors out: a funnel shape) and **two-way traders** (wholesalers who buy from and sell to the same partners: a loop shape).

In [ ]:
# How did shape alone do, and shape + evidence?
_, _, truth = lk.load(truth=True)
arch = truth.set_index("node_id").archetype
ev["planted"] = arch.reindex(ev.index).values
tab = pd.crosstab(ev["planted"], [ev["shape"], ev["review"]], margins=False)
tab.columns = [f"{s} | {'kept' if r else 'cleared'}" for s, r in tab.columns]
display(tab)

M["planted"] = arch.reindex(M.index).values
show = {"funnel": lk.PALETTE[1], "pass_through": lk.PALETTE[0], "round_trip": lk.PALETTE[7],
        "property_manager": lk.PALETTE[3], "two_way_trader": lk.PALETTE[2]}
fig = go.Figure(go.Scatter(x=M.x, y=M.y, mode="markers", name="everyone else",
                           marker=dict(size=5, color="#dcdbd5"), hoverinfo="skip"))
for k, c in show.items():
    m = M[M.planted == k]
    fig.add_trace(go.Scatter(x=m.x, y=m.y, mode="markers", name=k.replace("_", " "),
                             marker=dict(size=12, color=c, line=dict(width=1.5, color=lk.SURFACE)),
                             hovertext=[name[i] for i in m.index], hoverinfo="text"))
fig.update_layout(xaxis_title="fan-in ratio, log2", yaxis_title="overlap coefficient")
lk.style(fig, "The planted cases sit in the corners, next to their legitimate look-alikes", height=560)

## 5. Payee consensus: a two-hop check for vendor bank changes

A classic business email compromise: a payer receives "our bank details have changed", and the next payment goes to the fraudster. One payer's account view sees a normal payment to a known vendor.

The network sees **all** of the vendor's payers. A real bank change shows up for (nearly) all of them. A fraudulent one shows up for one.

In [ ]:
# Which accounts did each vendor's payers use, month by month?
vp = lk.load_vendor_payments()
use = vp.groupby(["vendor", "month", "vendor_account"]).payer.nunique().rename("payers").reset_index()
use["share_of_payers"] = use.payers / use.groupby(["vendor", "month"]).payers.transform("sum")
first_seen = use.groupby(["vendor", "vendor_account"]).month.min().rename("first_month")
use = use.merge(first_seen.reset_index(), on=["vendor", "vendor_account"])
new_acct = use[(use.month == use.first_month) & (use.first_month > use.month.min())]
new_acct = new_acct.assign(verdict=np.where(new_acct.share_of_payers >= 0.8, "consensus: real move",
                                            np.where(new_acct.share_of_payers <= 0.1, "one payer only: check before paying",
                                                     "partial: watch")))

vendors = sorted(vp.vendor.unique())
fig = make_subplots(rows=1, cols=len(vendors), subplot_titles=[f"{v} {name[v]}" for v in vendors], shared_yaxes=True)
acct_colors = {}
for j, v in enumerate(vendors):
    u = use[use.vendor == v]
    for acct in u.vendor_account.unique():
        acct_colors.setdefault(acct, lk.PALETTE[len(acct_colors) % len(lk.PALETTE)])
        a = u[u.vendor_account == acct]
        fig.add_trace(go.Bar(x=a.month, y=a.payers, name=acct, marker=dict(color=acct_colors[acct]),
                             showlegend=True), 1, j + 1)
fig.update_layout(barmode="stack")
display(lk.style(fig, "Payers per vendor account: a new account used by one payer stands out", height=430),
        new_acct[["vendor", "vendor_account", "month", "payers", "share_of_payers", "verdict"]].round(2))

In [ ]:
# The planted answers for the vendors
truth.set_index("node_id").loc[vendors, ["vendor_scenario"]]

## Takeaways

| | account view | network view |
|---|---|---|
| funnel | many small credits, one large debit | dozens of unrelated individuals feeding one payee, new pattern, swept every few days |
| pass-through | busy client, money in and out | money arrives and leaves within days, to different parties |
| round trip | two unrelated payments | same counterparty on both sides, same amount, days apart |
| vendor bank change | a normal payment to a known vendor | one payer out of sixty uses the new account |

* One-hop measures (degree, overlap, throughput) are cheap and find candidates.
* Timing, amounts and history clear the legitimate look-alikes.
* Never put real flagged customers in a training room: anything related to a suspicious-activity report is confidential by law.